# donut hunt, houston
a conference at the hilton americas, keynote at 08:45, and we want a real donut before it.
41 donut and kolache places from osm within ~16 km. which one, how do we get there, when do we leave, and are we back in time. (the plan said christy's, we ended up at voodoo.)


In [41]:
import os, json, pandas as pd, requests
os.environ.setdefault("VIBE_MAX_TOKENS", "64000")   # chained edits run long
import vibe_widget as vw
vw.config(model="anthropic/claude-opus-5")

Config(provider='openrouter', host='openrouter.ai', model='anthropic/claude-opus-5', key_source='OPENROUTER_API_KEY', environment='vscode-like', temperature=0.7, timeout=120.0, streaming=True, data_privacy='sample', sample_rows=3, mode='standard', theme=None, execution='auto', retry=2, agent_preset='project', agent_run=None, bypass_row_guard=False)

In [42]:
shops = pd.read_csv("data/houston_donuts.csv")  # overpass 2026-09-13; hours patched for 2 shops from their own listings
HOTEL = dict(name="Hilton Americas-Houston", lat=29.7522, lon=-95.3578)
shops.head(8)

,name,lat,lon,street,hours,chain,kolache,km_from_hotel,osm_id,note
0,Not Jus Donuts Cakes & More,29.74175,-95.35949,NaN,Tu-Fr 10:00-18:00; Sa 10:00-16:00,False,False,1.17,3710907490,NaN
1,BB Donuts,29.74421,-95.38797,Westheimer Road,NaN,False,False,3.04,10566816709,"listed as permanently closed (yelp, 2026-02); ..."
2,Christy's,29.75299,-95.39242,West Gray Street,Mo-Sa 04:00-14:00; Su 05:00-14:00,False,False,3.34,902286343,NaN
3,Shipley Do-Nuts,29.72184,-95.35216,NaN,NaN,True,False,3.42,364404158,NaN
4,Voodoo Doughnut,29.74475,-95.39387,Westheimer Road,NaN,True,False,3.58,428095827,NaN
5,Voodoo Doughnut,29.76935,-95.39835,Washington Avenue,24/7,True,False,4.35,13751666327,NaN
6,Shipley Do-Nuts,29.79348,-95.37610,NaN,NaN,True,False,4.92,1032364496,NaN
7,Shipley Do-Nuts,29.74896,-95.41033,South Shepherd Drive,NaN,True,False,5.08,9100306877,NaN


## 1. change the question

In [ ]:
# how far we are willing to go?
# WHERE dist(hotel) < r.   r = the circle. drag it.
m = vw.create(
    """leaflet map, 560px tall, esri light grey canvas basemap, of the donut shops. the hotel is a fixed ink pin at lat 29.7522 lon -95.3578.
    one circle centred on the hotel, radius 3.5 km to start; dragging anywhere on its edge changes the radius, a small label on the edge reads '<r> km'.
    shops inside the circle are solid dots (accent when chain is False, grey when True), shops outside are faded small dots.
    one count line top-left: '<n> inside · <k> independent'. hovering a shop shows name, street, hours.""",
    shops,
    outputs=vw.outputs(inside="row indices of shops inside the circle", radius_km="current radius in km"),
    theme="minimal",
)

In [44]:
# JOIN router.   straight-line km lies in houston. ask valhalla once per mode: minutes there and back, 10/20/30 min reach, the street route to every shop. cached to disk after the first run.
V = "https://valhalla1.openstreetmap.de"
COSTING = dict(walk="pedestrian", bike="bicycle", drive="auto")
pts = [dict(lat=r.lat, lon=r.lon) for r in shops.itertuples()]                  # pts[i] = shops.iloc[i]
def decode(s):                                                                   # valhalla polyline, 1e6 precision
    out, i, ll = [], 0, [0, 0]
    while i < len(s):
        for k in (0, 1):
            res = sh = 0
            while True:
                b = ord(s[i]) - 63; i += 1
                res |= (b & 0x1f) << sh; sh += 5
                if b < 0x20: break
            ll[k] += ~(res >> 1) if res & 1 else res >> 1
        out.append([round(ll[0] / 1e6, 5), round(ll[1] / 1e6, 5)])
    return out
def ask(costing):
    post = lambda ep, body: requests.post(f"{V}/{ep}", json=dict(body, costing=costing), timeout=60).json()
    back = [x[0]["time"] for x in post("sources_to_targets", {"sources": pts, "targets": [HOTEL]})["sources_to_targets"]]
    iso = post("isochrone", {"locations": [HOTEL], "contours": [{"time": 10}, {"time": 20}, {"time": 30}], "polygons": True})
    trips = [post("route", {"locations": [HOTEL, p]}).get("trip") for p in pts]
    there = [t["summary"]["time"] if t else None for t in trips]
    shapes = [decode(t["legs"][0]["shape"]) if t else [] for t in trips]
    return dict(there=there, back=back, iso=iso, shapes=shapes)                  # seconds; None = router couldn't snap that shop
if not os.path.exists("data/houston_routes.json"):
    json.dump({mode: ask(c) for mode, c in COSTING.items()}, open("data/houston_routes.json", "w"))
R = json.load(open("data/houston_routes.json"))
for mode, kmh in dict(walk=5, bike=15, drive=30).items():                        # fallback: straight line at a flat speed
    shops[f"{mode}_min"] = pd.Series(R[mode]["there"]).div(60).fillna(shops.km_from_hotel / kmh * 60).round()
    shops[f"{mode}_back"] = pd.Series(R[mode]["back"]).div(60).fillna(shops[f"{mode}_min"]).round()
reach = {mode: R[mode]["iso"] for mode in R}
routes = {mode: [s or [[HOTEL["lat"], HOTEL["lon"]], [p["lat"], p["lon"]]] for s, p in zip(R[mode]["shapes"], pts)] for mode in R}
shops.sort_values("bike_min")[["name", "km_from_hotel", "walk_min", "bike_min", "drive_min", "hours"]].head(8)

,name,km_from_hotel,walk_min,bike_min,drive_min,hours
0,Not Jus Donuts Cakes & More,1.17,29.0,9.0,5.0,Tu-Fr 10:00-18:00; Sa 10:00-16:00
1,BB Donuts,3.04,60.0,17.0,7.0,NaN
2,Christy's,3.34,61.0,17.0,7.0,Mo-Sa 04:00-14:00; Su 05:00-14:00
5,Voodoo Doughnut,4.35,66.0,18.0,8.0,24/7
3,Shipley Do-Nuts,3.42,58.0,19.0,8.0,NaN
4,Voodoo Doughnut,3.58,68.0,21.0,8.0,NaN
6,Shipley Do-Nuts,4.92,71.0,24.0,8.0,NaN
7,Shipley Do-Nuts,5.08,83.0,24.0,10.0,NaN


In [ ]:
# When are we leaving and when we may able to get back
# ... AND open_at(leave + minutes).   the dial is a time budget: leave, back by. the target's trip rides inside it; the map answers who's open when we get there.
m2 = m.edit(
    """keep the circle, its count line and the hover tooltips. add a 12-hour dial right of the map (a clock face: 12 at the top, then 3, 6, 9 clockwise, a tick every hour). two handles, independent of each other, neither ever moves the other, on two different radii so they can never overlap (LEAVE's handle sits on the rim, BACK BY's handle sits 14px outside the rim; a press within 18px of both centres grabs the handle whose centre is nearer the pointer, so they stay separately grabbable even at the same time of day): LEAVE (ink) is when we leave the hotel, start Tuesday 06:30 (the day row below starts on Tu); BACK BY (accent) is when we must be back at the hotel, start 08:45. a faint wedge between them is the time we have. hh:mm at each handle, outside the rim.
    clicking a shop makes it the target (thick ring; its name in one line under the dial); start with data row 2. the target's trip rides just inside the rim as a thin ink arc from LEAVE to LEAVE + bike_min + 10 + bike_back, with a small tick and hh:mm where we arrive at the shop; any part of the arc past BACK BY is red.
    under the dial one compact row of seven small day buttons Mo Tu We Th Fr Sa Su; left/right arrow keys nudge the last-touched handle 15 min while the dial has focus. while dragging, the pointer angle is measured from the centre of the dial svg's own getBoundingClientRect (the ref sits on the svg element itself, never on a wrapper that also holds text or buttons).
    draw input `reach` `.bike` (geojson: 30, 20, 10 minute bike reach from the hotel) as three translucent grey-blue bands under the shops at 6%, 9% and 12% opacity (darker the closer, never heavier than that even where they stack); the km circle stays on top.
    every shop inside the circle gets a small label 'hh:mm' = leave time + its `bike_min`. parse the osm `hours` text ('Mo-Sa 04:00-14:00; Su 05:00-14:00', '24/7', '05:00-20:00', 'Mo-Fr 05:00-21:00; Sa-Su 06:00-21:00', blank = unknown).
    open at that arrival time and day → solid dot and green label; closed at arrival → hollow ring and red label; unknown hours → dashed ring and grey label. a shop whose round trip (bike_min + 10 + bike_back) does not fit before BACK BY is drawn at 35% opacity.
    count line: '<n> inside · <k> independent · <j> open and back in time'. moving either handle, changing the day or the target restyles everything live. every new component takes React as a prop; never touch a module-level React.""",
    data=shops,
    inputs=vw.inputs(reach=reach),
    theme="minimal",
    outputs=vw.outputs(open_on_arrival="row indices inside the circle and open when we arrive", when="{day, hhmm} leave time, day 0=Mon..6=Sun, hhmm 'HH:MM'", back_by="'HH:MM' we must be back at the hotel", target="row index of the target shop"),   # edits keep earlier outputs
)

In [50]:
# GROUP BY region.   regions = whatever you draw.
m3 = m2.edit(
    """keep the circle, its count line, the reach bands, the arrival labels, the target and the dial with both handles (LEAVE on the rim, BACK BY outside it) exactly as they are. shift+drag on the map draws a lasso (turn off leaflet's shift box-zoom so the lasso gets the gesture; register mousedown on the map container with capture and mousemove/mouseup on window, right after the map is created and inside the same effect, and never re-create the map on re-render). each lasso is kept as a region named A, B, C... with a small draggable handle on every vertex; double-click inside a region deletes it.
    under the map a compact table, one row per region: name, n shops, n independent, n open on arrival, earliest arrival. hovering a row highlights its region. every new component takes React as a prop exactly like ClockDial does; never touch a module-level React.""",
    theme="minimal",
    outputs=vw.outputs(regions="dict region name -> list of row indices"),
)

## 2. show the change

In [ ]:
# HOW we go changes every number.   walk · bike · drive: the street route, the trip arc on the dial, every arrival label, and whether we're back in time.
m4 = m3.edit(
    """keep the circle, its count line, the reach bands, the arrival labels, the target, the dial with both handles and its day row, and the lasso regions with their table.
    add a three-way switch above the dial: walk · bike · drive, start on bike. the mode picks the data columns: minutes there = `<mode>_min`, minutes back = `<mode>_back`. on change: the reach bands switch to input `reach[mode]`, every arrival label, the trip arc and its arrival tick recompute, and so does the 35% fading of shops that do not fit before BACK BY.
    draw the street route hotel → target from input `routes[mode][row]` (list of [lat, lon]) as a 2px accent line above the bands, with a small '<n> min' label set beside the line at its midpoint on a white halo so it never covers the route or the hotel pin; it redraws when the target or the mode changes.
    under the dial two monospace lines that wrap rather than truncate: '06:30 → christy's 06:47, open · 10 min · back 07:18', then '<n> min before <BACK BY>' in green when back ≤ BACK BY or '<n> min after <BACK BY>' in red. a hollow tick on the rim at the back time. every new component, including the mode switch and the verdict lines, is declared as ({ React, ... }) and rendered with React={React}; never touch a module-level React.""",
    inputs=vw.inputs(routes=routes),
    theme="minimal",
    outputs=vw.outputs(mode="'walk' | 'bike' | 'drive'", back_hhmm="'HH:MM' we get back to the hotel", makes_it="true when back by BACK BY"),
)

In [36]:
m4.save("donut_hunt.vw")   # load elsewhere with vw.load(...); audit runs again on load

PosixPath('donut_hunt.vw')

## verdict

In [35]:
# plan said christy's: opens 4am, 17 min by bike. reality: we rode to voodoo on westheimer, it was open, it was good. the map got us to the right block; taste picked the door.
pick = shops[(shops.name == "Voodoo Doughnut") & (shops.street == "Westheimer Road")].iloc[0]
print(pick["name"], "·", pick.street, "·", pick.hours if isinstance(pick.hours, str) else "hours unknown in osm", "·", pick.bike_min, "min by bike")

Voodoo Doughnut · Westheimer Road · hours unknown in osm · 21.0 min by bike
